# GraviProb - Bayesian inference

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil*

In [ ]:
#r "../src/GraviNum/bin/Release/net10.0/Gravicode.Science.GraviNum.dll"
#r "../src/GraviFrame/bin/Release/net10.0/Gravicode.Science.GraviFrame.dll"
#r "../src/GraviProb/bin/Release/net10.0/Gravicode.Science.GraviProb.dll"
#r "nuget: ScottPlot, 5.1.59"

using Gravicode.Science.GraviFrame;
using Gravicode.Science.GraviNum;
using Gravicode.Science.GraviProb;
using Gravicode.Science.GraviProb.Models;

var flips = DataFrame.ReadCsv("../datasets/bayesian_coin.csv");
var trials = flips.RowCount;
var heads = (int)flips.Numeric("outcome").Sum();
Console.WriteLine($"{heads} heads in {trials} flips ({(double)heads / trials:P2})");

## The model

A uniform Beta(1,1) prior with a binomial likelihood. Because Beta is conjugate to the binomial, the exact posterior is available - which lets the sampler be checked against ground truth.

In [ ]:
var model = new BayesianModel()
    .AddDistribution("theta", Distribution.Beta(1, 1))
    .AddObservation("data", DistributionSpec.Binomial(trials, "theta"), heads);

var exact = Distribution.Beta(1, 1).PosteriorAfter(heads, trials - heads);
Console.WriteLine($"exact posterior: {exact.Name}, mean {exact.Mean:F6}, sd {exact.StandardDeviation:F6}");

## Sampling

In [ ]:
var posterior = model.SampleMCMC(iterations: 20_000, chains: 4, warmup: 10_000, seed: 42);
Console.Write(posterior.Summary());

var (low, high) = posterior.HighestDensityInterval("theta", 0.95);
Console.WriteLine($"95% HDI: [{low:F4}, {high:F4}]");
Console.WriteLine($"P(theta > 0.5) = {posterior["theta"].ToArray().Count(v => v > 0.5) / (double)posterior.TotalDraws:P2}");

## Posterior plot

In [ ]:
var draws = posterior["theta"];
var (edges, counts) = Statistics.Histogram(draws, bins: 60);
var width = edges[1] - edges[0];
var centres = new double[counts.Length];
var density = new double[counts.Length];
for (var i = 0; i < counts.Length; i++)
{
    centres[i] = (edges[i] + edges[i + 1]) / 2;
    density[i] = counts[i] / (draws.Size * width);
}

var curveX = NdArray.Linspace(0.01, 0.99, 400).ToArray();
var curveY = curveX.Select(exact.Density).ToArray();

var plot = new ScottPlot.Plot();
var bars = plot.Add.Bars(centres, density);
bars.LegendText = "MCMC draws";
foreach (var bar in bars.Bars) bar.Size = width * 0.9;

var curve = plot.Add.Scatter(curveX, curveY);
curve.LegendText = "exact Beta posterior"; curve.MarkerSize = 0; curve.LineWidth = 3;

var span = plot.Add.HorizontalSpan(low, high);
span.LegendText = "95% HDI";
span.FillColor = ScottPlot.Colors.Orange.WithAlpha(0.15);

plot.Title($"Posterior for theta: {heads} heads in {trials} flips");
plot.XLabel("theta"); plot.YLabel("density");
plot.ShowLegend();
plot.GetImageHtml(900, 550)

## Posterior predictive check

Can the fitted model reproduce the data it was fitted to?

In [ ]:
var replicated = posterior.PosteriorPredictive((v, rng) => rng.Binomial(trials, v["theta"]), draws: 5000, seed: 42);
Console.WriteLine($"replicated: mean {Statistics.Mean(replicated):F2}, sd {Statistics.Std(replicated):F2}");
Console.WriteLine($"observed {heads} sits at percentile {replicated.ToArray().Count(v => v < heads) / 5000.0:P1}");

## A Bayesian network

Explaining away: once the sprinkler accounts for the wet grass, rain becomes less necessary.

In [ ]:
var network = new BayesianNetwork()
    .AddVariable("rain", 0.8, 0.2)
    .AddVariable("sprinkler", 2, new[] { "rain" }, new[] { new[] { 0.6, 0.4 }, new[] { 0.99, 0.01 } })
    .AddVariable("wet", 2, new[] { "rain", "sprinkler" }, new[]
    {
        new[] { 1.00, 0.00 },
        new[] { 0.10, 0.90 },
        new[] { 0.20, 0.80 },
        new[] { 0.01, 0.99 },
    });

Console.WriteLine($"P(rain)                  = {network.Infer("rain")[1]:F4}");
Console.WriteLine($"P(rain | wet)            = {network.Infer("rain", new Dictionary<string, int> { ["wet"] = 1 })[1]:F4}");
Console.WriteLine($"P(rain | wet, sprinkler) = {network.Infer("rain", new Dictionary<string, int> { ["wet"] = 1, ["sprinkler"] = 1 })[1]:F4}");